In [1]:
import uuid
from typing import TypedDict
from langgraph.graph import StateGraph
from langgraph.constants import START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

In [2]:
class State(TypedDict):
    records: list[str]
    flagged: bool

In [3]:
def clean_data(state: State) -> State:
    cleaned = [r.strip() for r in state["records"] if r.strip()]
    return {**state, "records": cleaned}

def review_data(state: State) -> State:
    print("Manual review required.")
    answer = interrupt("Should we continue with these records?")
    print(f"Received input: {answer}")
    return state

def transform_data(state: State) -> State:
    transformed = [r.upper() for r in state["records"]]
    return {**state, "records": transformed}

In [4]:
subgraph_builder = StateGraph(State)

subgraph_builder.add_node("clean_data", clean_data)
subgraph_builder.add_node("review_data", review_data)
subgraph_builder.add_node("transform_data", transform_data)

subgraph_builder.add_edge(START, "clean_data")
subgraph_builder.add_edge("clean_data", "review_data")
subgraph_builder.add_edge("review_data", "transform_data")
subgraph_builder.add_edge("transform_data", END)

checkpointer = MemorySaver()

subgraph = subgraph_builder.compile(checkpointer=checkpointer)

In [5]:
def process_data(state: State) -> State:
    print("Processing data via subgraph.")
    updated_state = subgraph.invoke(state)
    return updated_state

def load_data(state: State) -> State:
    return {"records": ["  first entry ", "", " second entry "], "flagged": False}
    
def save_output(state: State) -> State:
    print(f"Final Records: {state['records']}")
    return state

In [6]:
builder = StateGraph(State)

builder.add_node("load_data", load_data)
builder.add_node("process_data", process_data)
builder.add_node("save_output", save_output)

builder.add_edge(START, "load_data")
builder.add_edge("load_data", "process_data")
builder.add_edge("process_data", "save_output")
builder.add_edge("save_output", END)

graph = builder.compile(checkpointer=checkpointer)

In [7]:
config = {
    "configurable": {
        "thread_id": uuid.uuid4()
    }
}

for step in graph.stream({"records": [], "flagged": False}, config=config):
    print(step)

{'load_data': {'records': ['  first entry ', '', ' second entry '], 'flagged': False}}
Processing data via subgraph.
Manual review required.
{'__interrupt__': (Interrupt(value='Should we continue with these records?', resumable=True, ns=['process_data:074437de-95bc-54f2-e832-3126036fafe6', 'review_data:01f9a7b2-b7fb-bd03-3419-f904ce69fd50']),)}


In [8]:
for step in graph.stream(Command(resume="yes"), config):
    print(step)

Processing data via subgraph.
Manual review required.
Received input: yes
{'process_data': {'records': ['FIRST ENTRY', 'SECOND ENTRY'], 'flagged': False}}
Final Records: ['FIRST ENTRY', 'SECOND ENTRY']
{'save_output': {'records': ['FIRST ENTRY', 'SECOND ENTRY'], 'flagged': False}}
